# RAG on HTML documents


## Step-1: Configuration

In [1]:
from my_config import MY_CONFIG

## Step-2: Setup Embeddings

In [2]:
# If connection to https://huggingface.co/ failed, uncomment the following path
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

In [3]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

Settings.embed_model = HuggingFaceEmbedding(
    model_name = MY_CONFIG.EMBEDDING_MODEL
)

## Step-3: Connect to Milvus

In [4]:
# connect to vector db
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.milvus import MilvusVectorStore

vector_store = MilvusVectorStore(
    uri = MY_CONFIG.DB_URI ,
    dim = MY_CONFIG.EMBEDDING_LENGTH , 
    collection_name = MY_CONFIG.COLLECTION_NAME,
    overwrite=False  # so we load the index from db
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print ("✅ Connected to Milvus instance: ", MY_CONFIG.DB_URI )

✅ Connected to Milvus instance:  ./rag_html.db


## Step-4: Load Document Index from DB

In [5]:
%%time

from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, storage_context=storage_context)

print ("✅ Loaded index from vector db:", MY_CONFIG.DB_URI )

✅ Loaded index from vector db: ./rag_html.db
CPU times: user 104 ms, sys: 13.6 ms, total: 118 ms
Wall time: 116 ms


## Step-5: Setup LLM

In [6]:
from llama_index.llms.replicate import Replicate
from llama_index.core import Settings

llm = Replicate(
    model= MY_CONFIG.LLM_MODEL,
    temperature=0.1
)

Settings.llm = llm

## Step-6: Query

In [7]:
query_engine = index.as_query_engine()
res = query_engine.query("What is AI Alliance?")
# res = query_engine.query("What is Apache license?")
print(res)

The AI Alliance is a collaborative initiative aimed at advancing artificial intelligence (AI) technology by fostering open innovation across various topical areas. It brings together leading developers, scientists, academics, students, business leaders, and partners from governments, non-profit, and civil society organizations. The Alliance focuses on improving foundational capabilities, safety, security, and trust in AI, while ensuring responsible and beneficial outcomes for people and society globally.

Key aspects of the AI Alliance include:

1. Formation of member-driven working groups across major AI topics.
2. Establishment of a governing board and technical oversight committee to advance project areas and set overall project standards.
3. A lightweight operating and governing structure that empowers individual collaborators and organizational members and sponsors to pursue projects according to their best fit.
4. Adherence to a community Code of Conduct, Competition Law Guidelin

In [8]:
query_engine = index.as_query_engine()
res = query_engine.query("Where was the demo night held?")
# res = query_engine.query("What are some Apache software?")
print(res)

The demo night was held at Shack15 in San Francisco, CA.


In [9]:
query_engine = index.as_query_engine()
res = query_engine.query("When was the moon landing?")
print(res)

The context information provided does not include any details about a moon landing. It only lists various events related to AI, data preparation, and meetups. Therefore, I cannot provide an answer to the query about the moon landing based on the given context.
